# PhysAR-IDS -- NB2a (v6): Optuna TPE tuning for the other seven models -- multiclass, multiseed

Models tuned here (XGBoost is NB2): **RandomForest, LogisticRegression, PerClassAutoencoder, LSTM, Transformer, GAT, TCN**.
(`GAT` = the *masked* GAT of NB1 v6. The original GAT scaled empty PLC nodes into non-zero vectors that attention then used as keys,
which NB1 section 13 diagnoses as a defect; tuning a model with a known defect would waste the budget. The unmasked GAT is not tuned.)

**Inputs:** `phase1_data_v6.pkl`, `plc_corr_matrix_train_only_v6.pkl`, `baseline_results_v6.pkl` (NB1 v6), and, optionally, `optuna_xgboost_results_v6.pkl` (NB2) in `SmartGrid_Dissertation`.

## What differs from the old v4 tuning notebook -- and why

| Old v4 NB2a | This notebook | Reason |
|---|---|---|
| Encoder trials (TCN/LSTM/Transformer/GAT) were **scored on the held-out TEST windows** | Every trial is scored by **causal, time-blocked CV inside the TRAIN data only**; the test set is read once, at the end | Selecting hyper-parameters on the test set inflates the reported score. This is the most important correction. |
| Tabular models scored on one 80/20 split | 3 expanding-window folds (fit 0-55% -> validate 55-70%, 0-70% -> 70-85%, 0-85% -> 85-100% of each `(class, PLC)` series) | One validation block is noisy. The same folds were used to choose the v6 feature set, so the protocol is already validated. A window/graph that straddles a fold boundary is dropped. |
| TCN/LSTM/Transformer/GAT architectures differed from NB1's (e.g. a different TCN block) | The **NB1 v6 architectures**, parameterised, with the NB1 defaults enqueued as trial 0 | A tuned model is only comparable to its baseline if it is the same model family. |
| One seed, one fit | Search on one seed; **final comparison tuned vs baseline on 5 seeds**, paired t-test | A single-seed gain is not evidence. |
| Baseline numbers taken from another run | The baseline is **re-fitted here, in the same pipeline and the same session**, as the tuned model | Removes pipeline and backend differences. NB1's numbers are printed for reference only. |
| 30 epochs fixed for the final fit, 15 in the search | `n_epochs` is a searched hyper-parameter, the same value is used in the search and in the final fit | A search at 15 epochs followed by a fit at 30 tunes a different model from the one reported. |

## Honest limits to carry into the dissertation
* The test set contains only **422 independent bursts** (97k rows are copies of them); the **GAT test set is 55 graphs**. Paired t-tests over 5 seeds measure seed noise, not test-set sampling noise.
* A CV gain smaller than the fold-to-fold spread is not distinguishable from noise; the notebook prints both.
* Encoder/AE searches use fewer folds than the tabular ones (configurable in `CFG`) because a single TCN fit took ~20 min in NB1. Fewer folds = noisier selection; the printed fold spread shows how much.
* If tuning does not beat the baseline for a model, that is the result and the dissertation should say so.

**Runtime guide (your GPU):** RF ~1-2 h, LR ~0.5 h, AE ~1-1.5 h, LSTM ~1 h, Transformer ~1.5-2 h, GAT ~0.5 h, TCN 3-5 h + final fits. Every model has a trial cap and a wall-clock timeout (`CFG`), studies are stored in an SQLite file so an interrupted run **resumes**, and results are checkpointed after each model. Run the model sections one at a time if you prefer; TCN is last.

## 0. Setup and configuration

In [ ]:
import os, sys, random, time, json, platform, subprocess, warnings, copy
import numpy as np
import pandas as pd
import joblib
from scipy import stats
warnings.filterwarnings('ignore')
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna']); import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
optuna.logging.set_verbosity(optuna.logging.WARNING)

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEARCH_SEED = 42
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False          # True = tiny plumbing run (2 trials, 2 seeds, subsampled data); never use for results
RUN_MODELS = ['RandomForest', 'LogisticRegression', 'PerClassAutoencoder', 'LSTM', 'Transformer', 'GAT', 'TCN']

# n_trials = cap, folds = which of the 3 expanding folds each trial uses, timeout_min = wall-clock cap for the search
CFG = {
    'RandomForest':        dict(n_trials=25, folds=[0, 1, 2], timeout_min=120),
    'LogisticRegression':  dict(n_trials=20, folds=[0, 1, 2], timeout_min=60),
    'PerClassAutoencoder': dict(n_trials=20, folds=[1, 2],    timeout_min=90),
    'LSTM':                dict(n_trials=30, folds=[0, 1, 2], timeout_min=90),
    'Transformer':         dict(n_trials=30, folds=[1, 2],    timeout_min=120),
    'GAT':                 dict(n_trials=40, folds=[0, 1, 2], timeout_min=30),
    'TCN':                 dict(n_trials=15, folds=[1],       timeout_min=240),
}
if SMOKE_TEST:
    SEEDS = SEEDS[:2]
    for k in CFG: CFG[k].update(n_trials=2, timeout_min=10)

def set_all_seeds(seed, deterministic=True):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic; torch.backends.cudnn.benchmark = not deterministic

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_all_seeds(SEARCH_SEED)
STORAGE = None if SMOKE_TEST else 'sqlite:///optuna_nb2a_v6.db'     # relative path -> SmartGrid_Dissertation
print(f'optuna {optuna.__version__} | torch {torch.__version__} | xgboost {xgb.__version__} | device {DEVICE}'
      + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
print(f'cwd: {os.getcwd()} | SMOKE_TEST={SMOKE_TEST} | seeds {SEEDS}')
if DEVICE.type != 'cuda': print('WARNING: no CUDA -- encoder searches will be very slow.')

## 1. Load NB1 v6 data (with version check)

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, (data.get('PIPELINE_VERSION'), data.get('TEST_FRAC'))
print(f"Version check passed: {PIPELINE_VERSION}, TEST_FRAC={data['TEST_FRAC']}, EMBARGO_SECONDS={data['EMBARGO_SECONDS']}")
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']).astype(int); y_test = np.asarray(data['y_test']).astype(int)
PLC_train = np.asarray(data['PLCID_train']).astype(int); PLC_test = np.asarray(data['PLCID_test']).astype(int)
ts_train = np.asarray(data['timestamp_train']); ts_test = np.asarray(data['timestamp_test'])
FEATURE_COLS = data['FEATURE_COLS']; N_FEAT = len(FEATURE_COLS)
corr_matrix = np.asarray(joblib.load(f'plc_corr_matrix_train_only_{PIPELINE_VERSION}.pkl')); assert corr_matrix.shape == (9, 9)
if SMOKE_TEST:       # keep time order, thin the rows (plumbing check only)
    _k = np.arange(0, len(X_train), 6); X_train, y_train, PLC_train, ts_train = X_train[_k], y_train[_k], PLC_train[_k], ts_train[_k]
    _k = np.arange(0, len(X_test), 6);  X_test, y_test, PLC_test, ts_test = X_test[_k], y_test[_k], PLC_test[_k], ts_test[_k]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
print(f'Train {X_train.shape} Test {X_test.shape} | features {N_FEAT}')
print('Train classes:', np.bincount(y_train), '| Test classes:', np.bincount(y_test))

## 2. Causal, time-blocked validation folds (train data only)

`frac` = position of a row in its `(class, PLC)` train series, 0 to 1 (rows sharing a timestamp share `frac`, so a burst never straddles a boundary).
Fold *k* fits on `frac < b_k` and validates on `c_k <= frac < d_k`, with `(b, c, d)` = (0.55, 0.55, 0.70), (0.70, 0.70, 0.85), (0.85, 0.85, 1.0).
Training only ever uses the past of the validation block. Windows and graphs carry the `frac` of their first and last row; one that crosses a boundary is excluded.

In [ ]:
FOLDS = [(0.55, 0.55, 0.70), (0.70, 0.70, 0.85), (0.85, 0.85, 1.0001)]

def series_frac(label, plc, ts):
    df = pd.DataFrame({'y': label, 'p': plc, 't': ts})
    g = df.groupby(['y', 'p'])['t']
    return ((df['t'] - g.transform('min')) / (g.transform('max') - g.transform('min') + 1e-9)).values

frac_train = series_frac(y_train, PLC_train, ts_train)

def row_masks(f):
    b, c, d = FOLDS[f]
    return frac_train < b, (frac_train >= c) & (frac_train < d)

def span_masks(fs, fe, f):
    """windows / graphs: fit = entirely before b; val = entirely inside [c, d)"""
    b, c, d = FOLDS[f]
    return fe < b, (fs >= c) & (fe < d)

for f in range(3):
    fit, val = row_masks(f)
    print(f'fold {f}: fit rows {fit.sum():>7}  val rows {val.sum():>6}  val class counts {np.bincount(y_train[val], minlength=6)}')

## 3. Windows (TCN / LSTM / Transformer) and graphs (GAT), built exactly as in NB1

Windows: length 50, stride 25, per `(class, PLC)` series in time order. Graphs: one per `(class, 5 s bin)`, 9 PLC nodes, node = mean of that PLC's rows in the bin, absent nodes exactly zero.
Features are the NB1-scaled rows (an absent node stays zero, which is what NB1's masked scaling does).

In [ ]:
WINDOW, STRIDE, BIN_S, N_PLCS = 50, 25, 5.0, 9

def build_windows(X, y, plc, ts, frac):
    key = pd.DataFrame({'y': y, 'p': plc}); Xs, ys, fss, fes = [], [], [], []
    for _, idx in key.groupby(['y', 'p']).indices.items():
        o = idx[np.argsort(ts[idx], kind='stable')]; n = len(o)
        if n < WINDOW: continue
        st = np.arange(0, n - WINDOW + 1, STRIDE); ii = st[:, None] + np.arange(WINDOW)[None, :]
        Xs.append(X[o][ii]); ys.append(np.full(len(st), y[o[0]])); fss.append(frac[o][st]); fes.append(frac[o][st + WINDOW - 1])
    return np.concatenate(Xs).astype(np.float32), np.concatenate(ys), np.concatenate(fss), np.concatenate(fes)

def build_graphs(X, y, plc, ts):
    Xs, ys, fss, fes = [], [], [], []
    for lab in np.unique(y):
        m = (y == lab); t = ts[m]; tmin, rng = t.min(), max(t.max() - t.min(), 1e-9)
        d = pd.DataFrame(X[m]); d['b'] = ((t - tmin) // BIN_S).astype(int); d['p'] = plc[m]
        means = d.groupby(['b', 'p']).mean()
        for b in sorted(d['b'].unique()):
            g = np.zeros((N_PLCS, X.shape[1]), dtype=np.float32); sub = means.loc[b]
            for p_ in sub.index: g[int(p_) - 1] = sub.loc[p_].values
            Xs.append(g); ys.append(lab); fss.append(b * BIN_S / rng); fes.append((b + 1) * BIN_S / rng)
    return np.stack(Xs), np.array(ys), np.array(fss), np.array(fes)

t0 = time.time()
Xw_tr, yw_tr, fsw_tr, few_tr = build_windows(X_train, y_train, PLC_train, ts_train, frac_train)
Xw_te, yw_te, _, _ = build_windows(X_test, y_test, PLC_test, ts_test, np.zeros(len(X_test)))
Xg_tr, yg_tr, fsg_tr, feg_tr = build_graphs(X_train, y_train, PLC_train, ts_train)
Xg_te, yg_te, _, _ = build_graphs(X_test, y_test, PLC_test, ts_test)
print(f'windows: train {Xw_tr.shape} test {Xw_te.shape} | graphs: train {Xg_tr.shape} test {Xg_te.shape}  ({time.time() - t0:.0f}s)')
print('window classes train/test:', np.bincount(yw_tr), np.bincount(yw_te))
print('graph  classes train/test:', np.bincount(yg_tr), np.bincount(yg_te))
for f in range(3):
    fw, vw = span_masks(fsw_tr, few_tr, f); fg, vg = span_masks(fsg_tr, feg_tr, f)
    print(f'fold {f}: windows fit {fw.sum()} val {vw.sum()} | graphs fit {fg.sum()} val {vg.sum()}')

## 4. Shared harness: metrics, encoders (NB1 architectures, parameterised), SupCon training, Optuna runner

In [ ]:
def run_metrics(y_true, y_pred, y_tr=None, p_tr=None):
    out = {'test_f1': f1_score(y_true, y_pred, average='weighted'), 'test_macro_f1': f1_score(y_true, y_pred, average='macro'),
           'per_class_f1': f1_score(y_true, y_pred, average=None, labels=range(6), zero_division=0)}
    b = (y_true == 0); a = ~b
    out['benign_fpr'] = float((y_pred[b] != 0).mean()) if b.any() else np.nan
    out['attack_miss_rate'] = float((y_pred[a] == 0).mean()) if a.any() else np.nan
    if y_tr is not None:
        out['train_f1'] = f1_score(y_tr, p_tr, average='weighted'); out['gap'] = out['train_f1'] - out['test_f1']
    return out

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__(); self.temperature = temperature
    def forward(self, emb, labels):
        emb = F.normalize(emb, dim=1); sim = emb @ emb.T / self.temperature
        labels = labels.view(-1, 1); mask = torch.eq(labels, labels.T).float()
        eye = torch.eye(mask.shape[0], device=emb.device); mask = mask - eye; sim = sim - eye * 1e9
        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
        pos = mask.sum(1); valid = pos > 0
        return (-(mask * log_prob).sum(1)[valid] / pos[valid]).mean()

class CausalConv1d(nn.Module):
    def __init__(self, i, o, k, d):
        super().__init__(); self.pad = (k - 1) * d; self.conv = nn.Conv1d(i, o, k, dilation=d, padding=self.pad)
    def forward(self, x):
        out = self.conv(x); return out[:, :, :-self.pad] if self.pad > 0 else out

class ResBlock(nn.Module):
    def __init__(self, i, o, k, d, dropout):
        super().__init__(); self.c1 = CausalConv1d(i, o, k, d); self.c2 = CausalConv1d(o, o, k, d)
        self.act = nn.GELU(); self.drop = nn.Dropout(dropout); self.down = nn.Conv1d(i, o, 1) if i != o else None
    def forward(self, x):
        res = x if self.down is None else self.down(x)
        out = self.drop(self.act(self.c1(x))); out = self.drop(self.act(self.c2(out))); return self.act(out + res)

KERNEL_SETS = {'3-7-15': (3, 7, 15), '3-5-7': (3, 5, 7), '3-3-3': (3, 3, 3)}
class TCNEncoder(nn.Module):                      # NB1 architecture; kernels '3-7-15' + hidden 64 + dropout 0.1 = NB1 default
    def __init__(self, n_feat, hidden=64, dropout=0.1, kernels='3-7-15'):
        super().__init__(); k = KERNEL_SETS[kernels]
        self.b1 = ResBlock(n_feat, hidden, k[0], 1, dropout); self.b2 = ResBlock(hidden, hidden, k[1], 2, dropout)
        self.b3 = ResBlock(hidden, hidden, k[2], 4, dropout); self.proj = nn.Linear(hidden, hidden)
    def forward(self, x):
        x = self.b3(self.b2(self.b1(x.transpose(1, 2)))); return self.proj(x[:, :, -1])

class LSTMEncoder(nn.Module):                     # n_layers=1 = NB1 default
    def __init__(self, n_feat, hidden=64, n_layers=1, dropout=0.1):
        super().__init__(); self.lstm = nn.LSTM(n_feat, hidden, n_layers, batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.proj = nn.Linear(hidden, hidden)
    def forward(self, x):
        _, (h, _) = self.lstm(x); return self.proj(h[-1])

class TransformerEncoder(nn.Module):              # hidden 64, heads 4, layers 2, ff 2x, dropout 0.1 = NB1 default
    def __init__(self, n_feat, hidden=64, n_heads=4, n_layers=2, dropout=0.1, ff_mult=2):
        super().__init__(); self.inp = nn.Linear(n_feat, hidden)
        layer = nn.TransformerEncoderLayer(d_model=hidden, nhead=n_heads, dim_feedforward=hidden * ff_mult, dropout=dropout, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, num_layers=n_layers); self.proj = nn.Linear(hidden, hidden)
    def forward(self, x):
        return self.proj(self.tr(self.inp(x)).mean(dim=1))

class GATLayerMasked(nn.Module):
    def __init__(self, i, o, dropout):
        super().__init__(); self.W = nn.Linear(i, o, bias=False); self.attn = nn.Linear(2 * o, 1); self.act = nn.GELU(); self.drop = nn.Dropout(dropout)
    def forward(self, x, adj_bias, present):
        h = self.W(x); n = h.shape[1]
        e = self.attn(torch.cat([h.unsqueeze(2).expand(-1, -1, n, -1), h.unsqueeze(1).expand(-1, n, -1, -1)], -1)).squeeze(-1) + adj_bias.unsqueeze(0)
        e = e.masked_fill(~present.unsqueeze(1), -1e9)
        return self.act(torch.einsum('bij,bjd->bid', self.drop(F.softmax(e, -1)), h))

class GATEncoderMasked(nn.Module):                # adj_scale 3.0, hidden 64, dropout 0.1 = NB1 default
    def __init__(self, n_feat, corr, hidden=64, dropout=0.1, adj_scale=3.0):
        super().__init__(); self.g1 = GATLayerMasked(n_feat, hidden, dropout); self.g2 = GATLayerMasked(hidden, hidden, dropout); self.proj = nn.Linear(hidden, hidden)
        c = corr.copy().astype(np.float32); np.fill_diagonal(c, c.max() if c.max() > 0 else 1.0)
        self.register_buffer('adj_bias', torch.from_numpy(c * adj_scale).float())
    def forward(self, x):
        present = x.abs().sum(-1) > 0
        h = self.g2(self.g1(x, self.adj_bias, present), self.adj_bias, present) * present.unsqueeze(-1)
        return self.proj(h.sum(1) / present.sum(1, keepdim=True).clamp(min=1))

def build_encoder(name, p):
    if name == 'TCN':         return TCNEncoder(N_FEAT, p['hidden'], p['dropout'], p['kernels'])
    if name == 'LSTM':        return LSTMEncoder(N_FEAT, p['hidden'], p['n_layers'], p['dropout'])
    if name == 'Transformer': return TransformerEncoder(N_FEAT, p['hidden'], p['n_heads'], p['n_layers'], p['dropout'], p['ff_mult'])
    if name == 'GAT':         return GATEncoderMasked(N_FEAT, corr_matrix, p['hidden'], p['dropout'], p['adj_scale'])

def train_encoder(enc, X, y, p, seed):
    """SupCon training, whole dataset resident on the device, seeded batch order."""
    enc.to(DEVICE); opt = torch.optim.Adam(enc.parameters(), lr=p['lr']); crit = SupConLoss(p['temperature'])
    Xt = torch.from_numpy(X).float().to(DEVICE); yt = torch.from_numpy(y).long().to(DEVICE)
    gen = torch.Generator(); gen.manual_seed(seed); n, bs = len(Xt), p['batch_size']
    for _ in range(p['n_epochs']):
        enc.train(); perm = torch.randperm(n, generator=gen).to(DEVICE)
        for i in range(0, n, bs):
            ib = perm[i:i + bs]
            if len(ib) < 2: continue
            opt.zero_grad(); crit(enc(Xt[ib]), yt[ib]).backward(); opt.step()
    return enc

def embed(enc, X, bs=512):
    enc.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs): out.append(enc(torch.from_numpy(X[i:i + bs]).float().to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

def make_head(p, seed):
    return xgb.XGBClassifier(n_estimators=p['head_n_estimators'], max_depth=p['head_max_depth'], learning_rate=p['head_lr'], objective='multi:softprob',
                             num_class=6, eval_metric='mlogloss', random_state=seed, n_jobs=-1, tree_method='hist')

def encoder_fit_predict(name, p, Xa, ya, Xb, seed, want_train_pred=False, keep=None):
    set_all_seeds(seed, deterministic=(ENC_DETERMINISTIC if name == 'TCN' else True))
    enc = train_encoder(build_encoder(name, p), Xa, ya, p, seed)
    ea, eb = embed(enc, Xa), embed(enc, Xb)
    head = make_head(p, seed); head.fit(ea, ya, verbose=False)
    if keep is not None:
        torch.save(enc.state_dict(), f'nb2a_{name}_tuned_encoder_{PIPELINE_VERSION}.pt'); head.save_model(f'nb2a_{name}_tuned_head_{PIPELINE_VERSION}.json')
    return head.predict(eb), (head.predict(ea) if want_train_pred else None)

ENC_DETERMINISTIC = True   # may be switched off for TCN by the speed probe (section 12)

def run_search(name, sample, baseline, score_fold, final_eval, keep_artifacts=None):
    cfg = CFG[name]; print(f'\n{"=" * 100}\n{name}: {cfg["n_trials"]} trials max, folds {cfg["folds"]}, timeout {cfg["timeout_min"]} min\n{"=" * 100}')
    def objective(trial):
        params = {**baseline['full'], **sample(trial)}; scores = []
        for k, f in enumerate(cfg['folds']):
            scores.append(score_fold(params, f)); trial.report(float(np.mean(scores)), step=k)
            if k < len(cfg['folds']) - 1 and trial.should_prune(): raise optuna.TrialPruned()
        trial.set_user_attr('fold_scores', [float(s) for s in scores]); return float(np.mean(scores))
    study = optuna.create_study(study_name=f'{name}_{PIPELINE_VERSION}', storage=STORAGE, load_if_exists=True, direction='maximize',
                                sampler=TPESampler(seed=SEARCH_SEED), pruner=MedianPruner(n_startup_trials=6, n_warmup_steps=0))
    if len(study.trials) == 0: study.enqueue_trial(baseline['search'])          # trial 0 = NB1 default configuration
    finished = sum(t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED) for t in study.trials)
    t0 = time.time()
    def cb(st, tr):
        print(f'  trial {tr.number:>3} {tr.state.name:<9} value={tr.value if tr.value is not None else float("nan"):.4f} best={st.best_value:.4f} elapsed={(time.time() - t0) / 60:.1f} min')
    study.optimize(objective, n_trials=max(cfg['n_trials'] - finished, 0), timeout=cfg['timeout_min'] * 60, callbacks=[cb])
    best = {**baseline['full'], **study.best_params}
    comp = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    t0_val = comp[0].value if comp and comp[0].number == 0 else np.nan
    print(f'\nSearch done ({len(comp)} complete, {len(study.trials) - len(comp)} pruned/other). Best CV F1 {study.best_value:.4f} | baseline-config CV F1 (trial 0) {t0_val:.4f}')
    print('Best params:', {k: (round(v, 5) if isinstance(v, float) else v) for k, v in study.best_params.items()})
    sd = np.std(study.best_trial.user_attrs['fold_scores'], ddof=1) if len(cfg['folds']) > 1 else np.nan
    vals = sorted([t.value for t in comp], reverse=True)
    print(f'Fold spread of best trial: {sd:.4f} | trials within 0.005 of best: {sum(v >= study.best_value - 0.005 for v in vals)} of {len(vals)} | CV gain over default: {study.best_value - t0_val:+.4f}')
    try: print('Param importance:', {k: round(v, 3) for k, v in optuna.importance.get_param_importances(study).items()})
    except Exception as e: print('importance n/a:', str(e)[:60])
    study.trials_dataframe(attrs=('number', 'value', 'state', 'params', 'user_attrs')).to_csv(f'nb2a_trials_{name}_{PIPELINE_VERSION}.csv', index=False)
    # held-out comparison, same pipeline, 5 seeds each; test read once
    res = {'baseline': {}, 'tuned': {}}
    for tag, params in [('baseline', baseline['full']), ('tuned', best)]:
        if name == 'LogisticRegression':
            m = final_eval(params, SEEDS[0], keep=(tag == 'tuned') and keep_artifacts); res[tag] = {s: m for s in SEEDS}; print(f'  {tag:<8} (deterministic) test={m["test_f1"]:.4f} train={m["train_f1"]:.4f}')
            continue
        for s in SEEDS:
            t1 = time.time(); m = final_eval(params, s, keep=(tag == 'tuned' and s == SEEDS[0]) and keep_artifacts)
            res[tag][s] = m; print(f'  {tag:<8} seed={s:<5} train={m["train_f1"]:.4f} test={m["test_f1"]:.4f} gap={m["gap"]:.4f} ({time.time() - t1:.0f}s)')
    return {'best_params': best, 'search_best_params': study.best_params, 'best_cv_f1': study.best_value, 'default_cv_f1': t0_val,
            'best_fold_scores': study.best_trial.user_attrs['fold_scores'], 'n_trials_done': len(study.trials), 'results': res, 'folds_used': cfg['folds']}

ALL = {}
def checkpoint():
    joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'CFG': CFG, 'models': ALL}, f'optuna_nb2a_results_{PIPELINE_VERSION}.pkl')
    json.dump({k: {kk: (vv if not hasattr(vv, 'item') else vv.item()) for kk, vv in v['best_params'].items()} for k, v in ALL.items()},
              open(f'nb2a_tuned_params_{PIPELINE_VERSION}.json', 'w'), indent=2, default=str)
print('harness ready')

## 5. Random Forest

NB1 default: 200 trees, depth 12, `min_samples_leaf=1`, `sqrt` features. Because each distinct measurement is repeated ~229x, leaf size is searched up to 200 and
bootstrap size (`max_samples`) down to 30% so the forest can regularise at the scale of a burst. Depth is capped at 24 to keep memory sane.

In [ ]:
def rf_make(p, seed):
    return RandomForestClassifier(n_estimators=p['n_estimators'], max_depth=p['max_depth'], min_samples_split=p['min_samples_split'],
                                  min_samples_leaf=p['min_samples_leaf'], max_features=p['max_features'],
                                  max_samples=(None if p['max_samples'] >= 0.999 else p['max_samples']), random_state=seed, n_jobs=-1)
RF_BASE = dict(n_estimators=200, max_depth=12, min_samples_split=2, min_samples_leaf=1, max_features='sqrt', max_samples=1.0)
def rf_sample(t):
    return dict(n_estimators=t.suggest_int('n_estimators', 100, 400), max_depth=t.suggest_int('max_depth', 6, 24),
                min_samples_split=t.suggest_int('min_samples_split', 2, 50), min_samples_leaf=t.suggest_int('min_samples_leaf', 1, 200, log=True),
                max_features=t.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5]), max_samples=t.suggest_float('max_samples', 0.3, 1.0))
def rf_score(p, f):
    fit, val = row_masks(f); m = rf_make(p, SEARCH_SEED).fit(X_train[fit], y_train[fit])
    return f1_score(y_train[val], m.predict(X_train[val]), average='weighted')
def rf_final(p, seed, keep=None):
    m = rf_make(p, seed).fit(X_train, y_train); return run_metrics(y_test, m.predict(X_test), y_train, m.predict(X_train))
if 'RandomForest' in RUN_MODELS:
    ALL['RandomForest'] = run_search('RandomForest', rf_sample, {'search': RF_BASE, 'full': RF_BASE}, rf_score, rf_final); checkpoint()

## 6. Logistic Regression

Only the regularisation strength `C` is searched (L2, lbfgs): the model is linear on 93 standardised features and the other solver options in the old notebook (saga, no penalty) change speed, not the function class.
LR is deterministic, so it is fitted once; there is no seed variance and no paired test, only a difference.

In [ ]:
LR_BASE = dict(C=1.0, max_iter=1000)
def lr_sample(t): return dict(C=t.suggest_float('C', 1e-4, 1e3, log=True))
def lr_score(p, f):
    fit, val = row_masks(f); m = LogisticRegression(C=p['C'], max_iter=p['max_iter'], random_state=SEARCH_SEED, n_jobs=-1).fit(X_train[fit], y_train[fit])
    return f1_score(y_train[val], m.predict(X_train[val]), average='weighted')
def lr_final(p, seed, keep=None):
    m = LogisticRegression(C=p['C'], max_iter=p['max_iter'], random_state=SEARCH_SEED, n_jobs=-1).fit(X_train, y_train)
    if keep: joblib.dump(m, f'nb2a_LogisticRegression_tuned_{PIPELINE_VERSION}.pkl')
    return run_metrics(y_test, m.predict(X_test), y_train, m.predict(X_train))
if 'LogisticRegression' in RUN_MODELS:
    ALL['LogisticRegression'] = run_search('LogisticRegression', lr_sample, {'search': dict(C=1.0), 'full': LR_BASE}, lr_score, lr_final, keep_artifacts=True); checkpoint()

## 7. Per-class autoencoder

One autoencoder per class on the rows of that class; a row is assigned to the class whose autoencoder reconstructs it best. Searched: bottleneck, hidden width, learning rate, epochs, batch size.
NB1 default: bottleneck 8, hidden 16, lr 1e-3, 30 epochs, batch 256.

In [ ]:
class ClassAutoencoder(nn.Module):
    def __init__(self, n, bottleneck, hidden):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.dec = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n))
    def forward(self, x): return self.dec(self.enc(x))

def train_ae(Xc, p, seed):
    torch.manual_seed(seed); gen = torch.Generator(); gen.manual_seed(seed)
    m = ClassAutoencoder(N_FEAT, p['bottleneck'], p['hidden']).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=p['lr']); crit = nn.MSELoss()
    Xt = torch.from_numpy(Xc).float().to(DEVICE); n = len(Xt)
    for _ in range(p['n_epochs']):
        m.train(); perm = torch.randperm(n, generator=gen).to(DEVICE)
        for i in range(0, n, p['batch_size']):
            xb = Xt[perm[i:i + p['batch_size']]]; opt.zero_grad(); crit(m(xb), xb).backward(); opt.step()
    return m

def recon_err(m, X, bs=8192):
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            b = torch.from_numpy(X[i:i + bs]).float().to(DEVICE); out.append(((m(b) - b) ** 2).mean(1).cpu().numpy())
    return np.concatenate(out)

def ae_predict(p, Xa, ya, Xbs, seed):
    aes = {c: train_ae(Xa[ya == c], p, seed + c) for c in range(6)}
    return aes, [np.stack([recon_err(aes[c], Xb) for c in range(6)], 1).argmin(1) for Xb in Xbs]

AE_BASE = dict(bottleneck=8, hidden=16, lr=1e-3, n_epochs=30, batch_size=256)
def ae_sample(t):
    return dict(bottleneck=t.suggest_int('bottleneck', 4, 32), hidden=t.suggest_int('hidden', 16, 128), lr=t.suggest_float('lr', 1e-4, 1e-2, log=True),
                n_epochs=t.suggest_int('n_epochs', 10, 50), batch_size=t.suggest_categorical('batch_size', [256, 512, 1024]))
def ae_score(p, f):
    fit, val = row_masks(f); _, (pv,) = ae_predict(p, X_train[fit], y_train[fit], [X_train[val]], SEARCH_SEED)
    return f1_score(y_train[val], pv, average='weighted')
def ae_final(p, seed, keep=None):
    aes, (pte, ptr) = ae_predict(p, X_train, y_train, [X_test, X_train], seed)
    if keep: torch.save({c: a.state_dict() for c, a in aes.items()}, f'nb2a_PerClassAutoencoder_tuned_{PIPELINE_VERSION}.pt')
    return run_metrics(y_test, pte, y_train, ptr)
if 'PerClassAutoencoder' in RUN_MODELS:
    ALL['PerClassAutoencoder'] = run_search('PerClassAutoencoder', ae_sample, {'search': AE_BASE, 'full': AE_BASE}, ae_score, ae_final, keep_artifacts=True); checkpoint()

## 8. Sequence / graph encoders: shared search spaces

Every encoder has the SupCon training knobs (lr, temperature, batch size, epochs) and the XGBoost head (trees, depth, learning rate), plus its own architecture knobs.
NB1 default = trial 0 of every study: lr 1e-3, temperature 0.1, batch 128 (GAT: 32), 30 epochs, head 100 trees / depth 6 / lr 0.1.

In [ ]:
def head_base(): return dict(head_n_estimators=100, head_max_depth=6, head_lr=0.1)
def train_knobs(t, batches):
    return dict(lr=t.suggest_float('lr', 1e-4, 1e-2, log=True), temperature=t.suggest_float('temperature', 0.05, 0.5),
                batch_size=t.suggest_categorical('batch_size', batches), n_epochs=t.suggest_categorical('n_epochs', [10, 20, 30, 40]),
                head_n_estimators=t.suggest_int('head_n_estimators', 50, 300), head_max_depth=t.suggest_int('head_max_depth', 3, 8),
                head_lr=t.suggest_float('head_lr', 0.03, 0.3, log=True))
TRAIN_BASE = dict(lr=1e-3, temperature=0.1, batch_size=128, n_epochs=30, **head_base())

def make_enc_funcs(name, kind):
    if kind == 'win': Xa_all, ya_all, fs, fe, Xte, yte = Xw_tr, yw_tr, fsw_tr, few_tr, Xw_te, yw_te
    else:             Xa_all, ya_all, fs, fe, Xte, yte = Xg_tr, yg_tr, fsg_tr, feg_tr, Xg_te, yg_te
    def score(p, f):
        fit, val = span_masks(fs, fe, f); pv, _ = encoder_fit_predict(name, p, Xa_all[fit], ya_all[fit], Xa_all[val], SEARCH_SEED)
        return f1_score(ya_all[val], pv, average='weighted')
    def final(p, seed, keep=None):
        pte, ptr = encoder_fit_predict(name, p, Xa_all, ya_all, Xte, seed, want_train_pred=True, keep=keep)
        return run_metrics(yte, pte, ya_all, ptr)
    return score, final

## 9. LSTM

In [ ]:
LSTM_BASE = dict(hidden=64, n_layers=1, dropout=0.1, **TRAIN_BASE)
def lstm_sample(t):
    return dict(hidden=t.suggest_int('hidden', 32, 128), n_layers=t.suggest_int('n_layers', 1, 3), dropout=t.suggest_float('dropout', 0.0, 0.3), **train_knobs(t, [64, 128, 256]))
if 'LSTM' in RUN_MODELS:
    _s, _f = make_enc_funcs('LSTM', 'win')
    ALL['LSTM'] = run_search('LSTM', lstm_sample, {'search': LSTM_BASE, 'full': LSTM_BASE}, _s, _f, keep_artifacts=True); checkpoint()

## 10. Transformer

In [ ]:
TR_BASE = dict(hidden=64, n_heads=4, n_layers=2, dropout=0.1, ff_mult=2, **TRAIN_BASE)
def tr_sample(t):
    return dict(hidden=t.suggest_categorical('hidden', [32, 64, 96, 128]), n_heads=t.suggest_categorical('n_heads', [2, 4, 8]), n_layers=t.suggest_int('n_layers', 1, 4),
                dropout=t.suggest_float('dropout', 0.0, 0.3), ff_mult=t.suggest_categorical('ff_mult', [1, 2, 4]), **train_knobs(t, [64, 128, 256]))
if 'Transformer' in RUN_MODELS:
    _s, _f = make_enc_funcs('Transformer', 'win')
    ALL['Transformer'] = run_search('Transformer', tr_sample, {'search': TR_BASE, 'full': TR_BASE}, _s, _f, keep_artifacts=True); checkpoint()

## 11. GAT (masked)

Only 257 train graphs and **55 test graphs**: the per-fold validation sets are ~40 graphs, so the selection is noisy and the seed-to-seed spread of the final scores is large.
Searched: width, dropout, adjacency scale (NB1 uses 3.0), plus the shared knobs. Treat any difference as indicative, not conclusive.

In [ ]:
GAT_BASE = dict(hidden=64, dropout=0.1, adj_scale=3.0, **{**TRAIN_BASE, 'batch_size': 32})
def gat_sample(t):
    return dict(hidden=t.suggest_int('hidden', 32, 128), dropout=t.suggest_float('dropout', 0.0, 0.3), adj_scale=t.suggest_float('adj_scale', 0.5, 8.0),
                **train_knobs(t, [16, 32, 64]))
if 'GAT' in RUN_MODELS:
    _s, _f = make_enc_funcs('GAT', 'graph')
    ALL['GAT'] = run_search('GAT', gat_sample, {'search': GAT_BASE, 'full': GAT_BASE}, _s, _f, keep_artifacts=True); checkpoint()

## 12. TCN (slowest -- run last)

In NB1 one TCN fit took ~20 min (the dilated causal convolutions are slow under `cudnn.deterministic=True`). The probe below times one epoch with and without deterministic cuDNN and uses the faster setting for **every** TCN fit in this
notebook (search, tuned and baseline alike), so the comparison stays like-for-like; a non-deterministic cuDNN means TCN results are reproducible only up to GPU rounding. The search uses a single fold (`CFG['TCN']['folds']`) and 15 trials.

In [ ]:
if 'TCN' in RUN_MODELS:
    Xp = Xw_tr[:4096]; yp = yw_tr[:4096]; tp = dict(TRAIN_BASE, n_epochs=1); timings = {}
    for det in ([True, False] if DEVICE.type == 'cuda' else [True]):
        set_all_seeds(0, deterministic=det); enc = TCNEncoder(N_FEAT, 64, 0.1, '3-7-15')
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        t1 = time.time(); train_encoder(enc, Xp, yp, tp, 0)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        timings[det] = time.time() - t1; print(f'  cudnn.deterministic={det}: {timings[det]:.2f}s per epoch on 4096 windows')
    ENC_DETERMINISTIC = (len(timings) == 1) or (timings[True] <= 1.3 * timings[False])
    print(f'-> TCN will use cudnn.deterministic={ENC_DETERMINISTIC}')

In [ ]:
TCN_BASE = dict(hidden=64, dropout=0.1, kernels='3-7-15', **TRAIN_BASE)
def tcn_sample(t):
    return dict(hidden=t.suggest_categorical('hidden', [32, 64, 96, 128]), dropout=t.suggest_float('dropout', 0.0, 0.3),
                kernels=t.suggest_categorical('kernels', ['3-7-15', '3-5-7', '3-3-3']), **train_knobs(t, [64, 128, 256]))
if 'TCN' in RUN_MODELS:
    _s, _f = make_enc_funcs('TCN', 'win')
    ALL['TCN'] = run_search('TCN', tcn_sample, {'search': TCN_BASE, 'full': TCN_BASE}, _s, _f, keep_artifacts=True); checkpoint()

## 13. Summary: tuned vs baseline (same pipeline, 5 seeds), plus the NB1 and NB2 references

In [ ]:
T4 = stats.t.ppf(0.975, df=len(SEEDS) - 1)
def arr(res, tag, key='test_f1'): return np.array([res[tag][s][key] for s in SEEDS])
try: nb1 = joblib.load(f'baseline_results_{PIPELINE_VERSION}.pkl')['multiseed']
except Exception as e: nb1 = {}; print('NB1 reference missing:', e)
NB1_KEY = {'GAT': 'GAT_masked'}
rows, verdicts = [], {}
for name, r in ALL.items():
    b, t_ = arr(r['results'], 'baseline'), arr(r['results'], 'tuned'); d = t_ - b; det = (name == 'LogisticRegression')
    p = np.nan if det or d.std(ddof=1) == 0 else stats.ttest_rel(t_, b).pvalue
    v = 'n/a (deterministic)' if det else ('TUNED BETTER' if (p < 0.05 and d.mean() > 0) else 'TUNED WORSE' if (p < 0.05 and d.mean() < 0) else 'NO SIGNIFICANT DIFFERENCE')
    verdicts[name] = v; ref = nb1.get(NB1_KEY.get(name, name), {}).get('test_f1', {})
    rows.append({'Model': name, 'Baseline (here)': f'{b.mean():.4f} +/- {b.std(ddof=1):.4f}', 'Tuned': f'{t_.mean():.4f} +/- {t_.std(ddof=1):.4f}',
                 'Diff': f'{d.mean():+.4f}', 'paired p': round(p, 4) if p == p else 'n/a', 'Verdict': v,
                 'NB1 ref': f'{ref["mean"]:.4f}' if ref else 'n/a', 'CV gain (search)': f'{r["best_cv_f1"] - r["default_cv_f1"]:+.4f}',
                 'CV fold sd': f'{np.std(r["best_fold_scores"], ddof=1):.4f}' if len(r['best_fold_scores']) > 1 else 'n/a (1 fold)',
                 'Tuned train-test gap': f'{arr(r["results"], "tuned", "gap").mean():.4f}', 'Benign FPR (tuned)': f'{arr(r["results"], "tuned", "benign_fpr").mean():.4f}'})
summ = pd.DataFrame(rows); print(summ.to_string(index=False)); summ.to_csv(f'nb2a_summary_{PIPELINE_VERSION}.csv', index=False)

print('\nPer-class F1, tuned (mean over seeds):')
print(pd.DataFrame({n: np.stack([ALL[n]['results']['tuned'][s]['per_class_f1'] for s in SEEDS]).mean(0) for n in ALL}, index=class_names).round(4).T.to_string())

# ranking after tuning, including NB2's tuned XGBoost when available
try:
    x = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl'); xt = np.array([x['results']['tuned'][s]['test_f1'] for s in SEEDS])
    print(f'\nTuned XGBoost (NB2, GPU): {xt.mean():.4f} +/- {xt.std(ddof=1):.4f}')
    rank = sorted([(n, arr(ALL[n]['results'], 'tuned').mean()) for n in ALL] + [('XGBoost', xt.mean())], key=lambda z: -z[1])
    print('Ranking after tuning (mean test F1):'); [print(f'  {i + 1}. {n:<22} {v:.4f}') for i, (n, v) in enumerate(rank)]
    for n in ALL:
        if n == 'LogisticRegression': continue
        d = arr(ALL[n]['results'], 'tuned') - xt; pv = stats.ttest_rel(arr(ALL[n]['results'], 'tuned'), xt).pvalue
        print(f'  {n:<22} vs tuned XGBoost: {d.mean():+.4f} (paired p={pv:.3f})')
    print('NB: NB2 ran on GPU XGBoost, the other models in this notebook use their own backends; the seed sets match.')
except Exception as e:
    print('\nTuned-XGBoost reference not available:', str(e)[:80])

## 14. Save

In [ ]:
checkpoint()
json.dump({'notebook': 'NB2a', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'cfg': CFG, 'device': str(DEVICE), 'torch': torch.__version__, 'xgboost': xgb.__version__,
           'optuna': optuna.__version__, 'python': platform.python_version(), 'tcn_cudnn_deterministic': ENC_DETERMINISTIC, 'smoke_test': SMOKE_TEST,
           'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb2a_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'optuna_nb2a_results_{PIPELINE_VERSION}.pkl', f'nb2a_tuned_params_{PIPELINE_VERSION}.json', f'nb2a_summary_{PIPELINE_VERSION}.csv', f'nb2a_run_manifest_{PIPELINE_VERSION}.json',
          'optuna_nb2a_v6.db'] + [f'nb2a_trials_{n}_{PIPELINE_VERSION}.csv' for n in ALL]:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)
print('\nRandom Forest is not saved as a model file (hundreds of MB); refit it from nb2a_tuned_params_v6.json when a downstream notebook needs it.')